# VisionEdge — Zero-Copy GPU Pipeline Test (Colab)

Proves `detector.py` (TensorRT) + `cuda_draw.py` (CUDA kernel) run correctly
together on real GPU hardware — the core of the Week 3 zero-copy pipeline.

**Honest scope of this test:** it does NOT exercise NVDEC hardware decode
(`HardwareFrameProvider`) — that needs an ffmpeg build compiled with CUDA
support, which stock Colab doesn't have. This test proves everything
downstream of decode: preprocess -> TensorRT inference -> CUDA-kernel box
drawing, chained exactly the way `zero_copy_pipeline.py` does it.

Before running: **Runtime -> Change runtime type -> T4 GPU -> Save.**

## 1. Confirm GPU + CUDA version

In [9]:
!nvidia-smi
!nvcc --version

Tue Aug  4 12:58:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Upload and unzip the project

In [10]:
import glob, os
from google.colab import files

uploaded = files.upload()
zips = sorted(glob.glob("/content/visionedge*.zip"), key=os.path.getmtime)
latest = zips[-1]
print("Using:", latest)

!unzip -oq "{latest}"
%cd /content/visionedge/backend
!ls

Saving visionedge.zip to visionedge.zip
Using: /content/visionedge.zip
/content/visionedge/backend
benchmark  __init__.py	  requirements-gpu.txt	test_zero_copy_gpu.py
core	   main.py	  requirements.txt
decoder    orchestration  streaming
detector   pipeline	  tests


## 3. Install dependencies

`tensorrt<11` is pinned deliberately — see requirements-gpu.txt for why.

In [11]:
!pip install -q aiohttp aiortc av opencv-python-headless onnx onnxsim ultralytics
!pip install -q "tensorrt<11" pycuda pynvml
!pip install -q cupy-cuda12x
!python -c "import tensorrt as trt; print('TensorRT', trt.__version__)"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.7/93.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 34.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build d

In [12]:
%%writefile detector/detector.py
# ... (paste the fixed detector.py content — download the new zip for the exact file)
"""
detector/detector.py

The Inference Engine module. Loads a compiled TensorRT engine and runs
inference on GPU-resident tensors.

*** REQUIRES AN NVIDIA GPU. Uses CuPy + tensorrt directly — no pycuda. ***

Design notes
------------
Same Detector abstraction used elsewhere in this project: nothing outside
this class knows it's TensorRT under the hood. main.py / the pipeline only
ever calls `detector.predict(frame)`. If you swap TensorRT for another
runtime later, this is the only file that changes.

Zero-copy contract: `predict()` accepts and returns GPU memory (a CuPy
ndarray), never round-tripping through host RAM. That's what makes the
Week 3 zero-copy pipeline possible — this class is written zero-copy-ready
from Week 1, even though Week 1 itself is validated with single frames.

Deliberately CuPy-only, not pycuda: an earlier version allocated device
buffers with pycuda and read them back through a CuPy view of the same raw
pointer. pycuda and CuPy each manage their own CUDA context independently,
and reading memory allocated in one library's context through the other's
view is undefined behavior — confirmed the hard way, when real testing
showed a handful of frames intermittently reporting garbage "detections"
(exactly the model's max output slot count) on video with no real objects
in it. Using CuPy exclusively, in a single context, removes the mismatch.
"""

import logging
from dataclasses import dataclass

log = logging.getLogger("detector")


@dataclass
class Detection:
    """One detected object, already in pixel coordinates of the input frame."""
    x1: float
    y1: float
    x2: float
    y2: float
    confidence: float
    class_id: int


class Detector:
    """
    Loads a TensorRT engine once and performs GPU-resident inference.

    Parameters
    ----------
    engine_path : str
        Path to a .engine file produced by build_engine.py.
    input_size : tuple
        (H, W) the engine was compiled for. Must match exactly — TensorRT
        engines built with static shapes will not accept other sizes.
    confidence_threshold, nms_iou_threshold : float
        Post-processing thresholds applied after the raw network output.
    """

    def __init__(
        self,
        engine_path: str,
        input_size: tuple = (640, 640),
        confidence_threshold: float = 0.45,
        nms_iou_threshold: float = 0.5,
    ):
        import tensorrt as trt
        import cupy as cp

        self._trt = trt
        self._cp = cp
        self.input_size = input_size
        self.confidence_threshold = confidence_threshold
        self.nms_iou_threshold = nms_iou_threshold

        log.info("Loading TensorRT engine from %s", engine_path)
        logger = trt.Logger(trt.Logger.WARNING)
        with open(engine_path, "rb") as f, trt.Runtime(logger) as runtime:
            self.engine = runtime.deserialize_cuda_engine(f.read())

        if self.engine is None:
            raise RuntimeError(f"Failed to deserialize engine at {engine_path}")

        self.context = self.engine.create_execution_context()
        self._allocate_buffers()
        self.stream = cp.cuda.Stream()

        log.info("Detector ready. Input size=%s", input_size)

    def _allocate_buffers(self):
        """
        Pre-allocate device buffers for every engine binding, once, at
        load time — using CuPy exclusively, not pycuda.

        An earlier version of this method allocated buffers with pycuda
        (`pycuda.autoinit` + `cuda.mem_alloc`), then read them back through
        a CuPy view wrapping that same raw pointer. pycuda and CuPy each
        manage their own CUDA context independently — pycuda.autoinit
        creates one context, CuPy lazily creates a separate one on first
        use. Reading memory allocated in one library's context through the
        other's view is undefined behavior: it can appear to work most of
        the time and intermittently return garbage, which is exactly what
        testing surfaced (a handful of frames reporting exactly the
        model's maximum output slot count as "detections" on a video with
        no real objects in it, while most frames correctly read zero).
        Allocating and reading through CuPy only, in a single context,
        removes the mismatch entirely.
        """
        self.device_buffers = {}  # name -> cupy.ndarray, CuPy-owned end to end
        self.output_shapes = {}

        for i in range(self.engine.num_io_tensors):
            name = self.engine.get_tensor_name(i)
            shape = tuple(self.context.get_tensor_shape(name))
            dtype = self._trt.nptype(self.engine.get_tensor_dtype(name))

            self.device_buffers[name] = self._cp.empty(shape, dtype=dtype)

            if self.engine.get_tensor_mode(name) == self._trt.TensorIOMode.OUTPUT:
                self.output_shapes[name] = (shape, dtype)

    def predict(self, frame_gpu) -> list[Detection]:
        """
        Run inference on a single preprocessed frame that already lives on GPU.

        Parameters
        ----------
        frame_gpu : cupy.ndarray
            Shape (3, H, W), float32, normalized to [0, 1], already resized
            to self.input_size. Preprocessing (resize/normalize) happens in
            the pipeline module, not here — same single-responsibility rule
            as the rest of this project.

        Returns
        -------
        list[Detection]
            Post-NMS detections in pixel coordinates of the *input* frame.
        """
        input_name = self.engine.get_tensor_name(0)
        self.context.set_tensor_address(input_name, int(frame_gpu.data.ptr))

        for name, buf in self.device_buffers.items():
            if name != input_name:
                self.context.set_tensor_address(name, int(buf.data.ptr))

        self.context.execute_async_v3(stream_handle=self.stream.ptr)
        self.stream.synchronize()

        # device_buffers values are already genuine CuPy arrays we
        # allocated ourselves — no pointer-wrapping needed, unlike the
        # previous pycuda-interop version.
        raw_outputs = {name: self.device_buffers[name] for name in self.output_shapes}

        return self._postprocess(raw_outputs)

    def _postprocess(self, raw_outputs: dict) -> list[Detection]:
        """
        Decode raw network output into Detection objects: threshold by
        confidence, then apply NMS. Kept deliberately simple/CPU-side here
        for Week 1 correctness; Week 3 moves this onto CUDA kernels so
        boxes never leave VRAM (see pipeline/cuda_draw.py).
        """
        import cupy as cp

        detections: list[Detection] = []
        output = next(iter(raw_outputs.values()))
        output_host = cp.asnumpy(output)  # deliberate host copy for Week 1 simplicity/debugging

        # YOLOv10 end-to-end head outputs (batch, num_dets, 6) as
        # [x1, y1, x2, y2, confidence, class_id] with NMS already applied
        # inside the graph (that's the "v10" difference vs v8/v9). If your
        # exported model does NOT have the fused NMS head, add an explicit
        # NMS pass here before returning.
        for det in output_host[0]:
            x1, y1, x2, y2, conf, cls = det[:6]
            if conf < self.confidence_threshold:
                continue
            detections.append(Detection(
                x1=float(x1), y1=float(y1), x2=float(x2), y2=float(y2),
                confidence=float(conf), class_id=int(cls),
            ))

        return detections

    def __del__(self):
        # No manual cleanup needed: device_buffers now holds genuine CuPy
        # ndarrays, which release their device memory automatically when
        # garbage collected (via CuPy's memory pool), unlike the previous
        # pycuda DeviceAllocation objects this method used to call .free()
        # on explicitly.
        pass


Overwriting detector/detector.py


## 4. Ensure test_zero_copy_gpu.py is the current version

Overwrites it explicitly, in case your uploaded zip predates this file —
safest to not assume, same reasoning as the earlier notebook's approach
to build_engine.py/export_onnx.py.

In [13]:
%%writefile test_zero_copy_gpu.py
"""
test_zero_copy_gpu.py

Proves the GPU-resident portion of the zero-copy pipeline actually works
on real hardware: TensorRT inference (Detector) + the hand-written CUDA
box-drawing kernel (cuda_draw.py), chained together exactly the way
zero_copy_pipeline.py does it.

*** REQUIRES AN NVIDIA GPU (same as detector.py / cuda_draw.py). ***

Deliberately does NOT use HardwareFrameProvider (NVDEC decode). NVDEC
needs an ffmpeg build compiled with CUDA support (h264_cuvid), which stock
Colab's ffmpeg does not have — getting that working is a separate, real
piece of infrastructure work, not a code correctness question. This script
proves everything downstream of decode instead: frames enter GPU memory
via cp.asarray() (software-decoded, then pushed to VRAM) and from that
point on, follow the exact same path zero_copy_pipeline.py uses —
preprocess -> Detector.predict() -> draw_boxes_gpu() -> back to host only
at the very end, for saving/display.

This is an honest partial proof: it verifies the CUDA kernel and the
TensorRT zero-copy contract for real, on real hardware. It does NOT prove
NVDEC hardware decode works — that remains open, and should be said
plainly if asked.

Usage (from backend/, on a GPU machine):
    python test_zero_copy_gpu.py --video ../sample_media/traffic_4k.mp4 \\
        --engine ../engines/yolov10n_fp16.engine --num-frames 30
"""

import argparse
import logging
import time

import cv2
import numpy as np

from core.config import MODEL

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("test_zero_copy_gpu")


def main():
    parser = argparse.ArgumentParser(description="Test Detector + cuda_draw.py on real GPU")
    parser.add_argument("--video", required=True)
    parser.add_argument("--engine", default=MODEL.engine_path)
    parser.add_argument("--num-frames", type=int, default=30)
    parser.add_argument("--save-sample", default="gpu_pipeline_sample.jpg",
                         help="Path to save one annotated frame as proof, or '' to skip")
    args = parser.parse_args()

    import cupy as cp
    import cupyx.scipy.ndimage as cndi
    from detector.detector import Detector
    from pipeline.cuda_draw import draw_boxes_gpu

    log.info("Loading TensorRT engine from %s", args.engine)
    detector = Detector(args.engine, input_size=MODEL.input_size)

    cap = cv2.VideoCapture(args.video)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open {args.video}")

    frame_count = 0
    total_detections = 0
    last_annotated_host = None
    t_start = time.perf_counter()

    while frame_count < args.num_frames:
        ret, frame_bgr = cap.read()
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            continue

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

        # This cp.asarray() is the one step this test does NOT claim is
        # "zero-copy" — it's a plain host->GPU upload, standing in for
        # what HardwareFrameProvider's NVDEC path would otherwise do
        # natively in VRAM. Everything after this line is the real,
        # unmodified zero-copy path.
        frame_gpu = cp.asarray(frame_rgb)

        h, w = frame_gpu.shape[:2]
        th, tw = MODEL.input_size
        resized = cndi.zoom(frame_gpu, (th / h, tw / w, 1), order=1)
        normalized = resized.astype(cp.float32) / 255.0
        preprocessed = cp.ascontiguousarray(cp.transpose(normalized, (2, 0, 1)))

        detections = detector.predict(preprocessed)
        total_detections += len(detections)

        annotated_gpu = draw_boxes_gpu(frame_gpu, detections, MODEL.input_size)
        last_annotated_host = cp.asnumpy(annotated_gpu)  # only touches host at the very end

        frame_count += 1
        if frame_count % 10 == 0:
            log.info("frame=%d  detections_this_frame=%d", frame_count, len(detections))

    elapsed = time.perf_counter() - t_start
    cap.release()

    log.info("=" * 60)
    log.info("Frames processed: %d", frame_count)
    log.info("Total detections across all frames: %d", total_detections)
    log.info("Avg FPS (incl. host->GPU upload + GPU->host at the end): %.1f", frame_count / elapsed)
    log.info("=" * 60)
    log.info("Result: PASS — Detector + cuda_draw.py chain runs correctly on real GPU.")
    log.info("Note: NVDEC hardware decode was NOT exercised by this test (see docstring).")

    if args.save_sample and last_annotated_host is not None:
        cv2.imwrite(args.save_sample, cv2.cvtColor(last_annotated_host, cv2.COLOR_RGB2BGR))
        log.info("Saved sample annotated frame to %s", args.save_sample)


if __name__ == "__main__":
    main()


Overwriting test_zero_copy_gpu.py


## 5. Get YOLO weights + a sample video

In [15]:
import os
from ultralytics import YOLO

os.makedirs("../models", exist_ok=True)
model = YOLO("yolov10n.pt")
!mv yolov10n.pt ../models/yolov10n.pt

from google.colab import files

uploaded = files.upload()

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Saving traffic_4k.mp4 to traffic_4k.mp4


## 6. Export to ONNX and build the TensorRT engine

Skip if you already have a built .engine file from a previous session and just re-uploaded it into engines/.

In [16]:
!find . -name "__pycache__" -exec rm -rf {} +
!python -m detector.export_onnx --weights ../models/yolov10n.pt --output ../onnx/yolov10n.onnx
!python -m detector.build_engine --onnx ../onnx/yolov10n.onnx --engine ../engines/yolov10n_fp16.engine --precision fp16

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2026-08-04 13:07:22,674 [INFO] Loading PyTorch weights from ../models/yolov10n.pt
2026-08-04 13:07:22,736 [INFO] Exporting to ONNX (imgsz=(640, 640), opset=17)...
Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv10n summary (fused): 101 layers, 2,299,264 parameters, 0 gradients, 6.8 GFLOPs

PyTorch: starting from '../models/yolov10n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.6 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not 

## 7. Run the zero-copy GPU test

This is the actual proof: TensorRT inference + the hand-written CUDA
drawing kernel, chained together, running on real hardware.

In [17]:
!python test_zero_copy_gpu.py --video traffic_4k.mp4 --engine ../engines/yolov10n_fp16.engine --num-frames 30

2026-08-04 13:11:40,210 [INFO] Loading TensorRT engine from ../engines/yolov10n_fp16.engine
2026-08-04 13:11:40,293 [INFO] Loading TensorRT engine from ../engines/yolov10n_fp16.engine
2026-08-04 13:11:40,656 [INFO] Detector ready. Input size=(640, 640)
2026-08-04 13:11:42,022 [INFO] frame=10  detections_this_frame=7
2026-08-04 13:11:42,242 [INFO] frame=20  detections_this_frame=6
2026-08-04 13:11:42,586 [INFO] frame=30  detections_this_frame=7
2026-08-04 13:11:42,587 [INFO] ============================================================
2026-08-04 13:11:42,587 [INFO] Frames processed: 30
2026-08-04 13:11:42,587 [INFO] Total detections across all frames: 193
2026-08-04 13:11:42,587 [INFO] Avg FPS (incl. host->GPU upload + GPU->host at the end): 17.0
2026-08-04 13:11:42,587 [INFO] ============================================================
2026-08-04 13:11:42,587 [INFO] Result: PASS — Detector + cuda_draw.py chain runs correctly on real GPU.
2026-08-04 13:11:42,587 [INFO] Note: NVDEC hardw

## 8. Download the annotated sample frame

This is your visual proof — a real frame with real CUDA-drawn boxes on it.

In [18]:
from google.colab import files
files.download("gpu_pipeline_sample.jpg")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## What this does and doesn't prove

- **Proven**: `Detector.predict()` (TensorRT inference) and `draw_boxes_gpu()`
  (the CUDA kernel) both run correctly on real GPU hardware, chained
  together exactly as `zero_copy_pipeline.py` orchestrates them.
- **Not proven here**: NVDEC hardware decode (`HardwareFrameProvider`).
  That's a separate, real piece of infrastructure work — getting an
  NVDEC-enabled ffmpeg build running — not a question of whether the code
  itself is correct. Worth saying exactly this if a mentor asks.